<a href="https://colab.research.google.com/github/ArthiNelwadkar/EV-Data-Analysis/blob/main/Data_Cleaning_EV_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1: How many missing values exist in the dataset, and in which columns?

import pandas as pd
import numpy as np

# Load your dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Count missing values per column
missing_counts = df.isnull().sum()

# Show only columns with missing values
missing_counts[missing_counts > 0]



,0
County,12
City,12
Postal Code,12
Electric Range,12
Legislative District,703
Vehicle Location,20
Electric Utility,12
2020 Census Tract,12


In [ ]:
#2:How should missing or zero values in the Base MSRP and Electric Range columns be handled?

import pandas as pd
import numpy as np

# Step 1: Load dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Step 2: Check missing and zero values
print(df['Electric Range'].isnull().sum())
print((df['Electric Range'] == 0).sum())

# Step 3: Replace zeros with NaN (missing)
df['Electric Range'] = df['Electric Range'].replace(0, np.nan)

# Step 4: Impute missing values
# For BEVs (Battery Electric Vehicles), fill with median range by Model-Year
df.loc[df['Electric Vehicle Type'] == 'BEV', 'Electric Range'] = (
    df.groupby(['Model', 'Model Year'])['Electric Range']
      .transform(lambda x: x.fillna(x.median()))
)

# Step 5: Keep zeros for PHEVs (Plug-in Hybrids) if they truly have low range
# Verify cleaning
df['Electric Range'].describe()

Object `handled` not found.
12
179538


,Electric Range
count,101283.000000
mean,107.461440
std,97.180717
min,1.000000
25%,30.000000
50%,47.000000
75%,215.000000
max,337.000000


In [ ]:
#3: Are there duplicate records in the dataset? If so, how should they be managed?

import pandas as pd

# Step 1: Load dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Step 2: Check for duplicate rows
duplicates = df.duplicated()
print("Number of duplicate records:", duplicates.sum())

# Step 3: View duplicate examples (optional)
df[duplicates].head()

# Step 4: Remove duplicates
df.drop_duplicates(inplace=True)

# Step 5: Verify removal
print("After cleaning:", df.duplicated().sum())

Object `managed` not found.
Number of duplicate records: 0
After cleaning: 0


In [ ]:
#4: How can VINs be anonymized while maintaining uniqueness?

# Step 1: Import libraries
import pandas as pd
import hashlib

# Step 2: Load dataset
df = pd.read_csv("Electric_Vehicle_Population_Data.csv")

# Step 3: Inspect VIN column name
print(df.columns.tolist())   # confirm exact name, e.g. 'VIN (1-10)'

# Step 4: Define anonymization function (hashing)
def anonymize_vin(vin):
    return hashlib.sha256(vin.encode()).hexdigest()[:10]  # short unique code

# Step 5: Apply anonymization
df['VIN_Anon'] = df['VIN (1-10)'].apply(anonymize_vin)

# Step 6: Verify uniqueness
print("Unique VINs:", df['VIN (1-10)'].nunique())
print("Unique Anonymized VINs:", df['VIN_Anon'].nunique())

# Step 7: Drop original VIN column before sharing
df = df.drop(columns=['VIN (1-10)'])

# Step 8: Save cleaned dataset
df.to_csv("EV_Population_Anonymized.csv", index=False)


Object `uniqueness` not found.
['VIN (1-10)', 'County', 'City', 'State', 'Postal Code', 'Model Year', 'Make', 'Model', 'Electric Vehicle Type', 'Clean Alternative Fuel Vehicle (CAFV) Eligibility', 'Electric Range', 'Legislative District', 'DOL Vehicle ID', 'Vehicle Location', 'Electric Utility', '2020 Census Tract']
Unique VINs: 17281
Unique Anonymized VINs: 17281


In [1]:
#5 :How can Vehicle Location (GPS coordinates) be cleaned or converted for better readability?

import pandas as pd
from geopy.geocoders import Nominatim

# Step 1: Example dataset with messy GPS
df = pd.DataFrame({
    'Vehicle_Location': ['47.6062, -122.3321', '40.7128,-74.0060', '34.0522 , -118.2437']
})

# Step 2: Clean coordinates (remove spaces, split into lat/lon)
df[['Latitude','Longitude']] = df['Vehicle_Location'].str.replace(' ', '').str.split(',', expand=True)
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Step 3: Convert to readable location (city/state)
geolocator = Nominatim(user_agent="ev_location_cleaner")
def get_city(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en')
        return location.address.split(',')[0]  # short readable name
    except:
        return "Unknown"

df['Readable_Location'] = df.apply(lambda row: get_city(row['Latitude'], row['Longitude']), axis=1)

print(df)


      Vehicle_Location  Latitude  Longitude  \
0   47.6062, -122.3321   47.6062  -122.3321   
1     40.7128,-74.0060   40.7128   -74.0060   
2  34.0522 , -118.2437   34.0522  -118.2437   

                            Readable_Location  
0                                   Schuchart  
1                          New York City Hall  
2  Los Angeles Police Department Headquarters  
